# Aula 06 — nn.Module, parâmetros e buffers

Este laboratório investiga **registro de estado**, sem treinamento completo. Vamos comparar uma MLP explícita com a mesma função empacotada em módulos e provocar falhas que não impedem o backward, mas escondem pesos do registro.

Requer Python >=3.10, PyTorch >=2.6 e NumPy >=1.24. Validado em Python 3.12.14, PyTorch 2.6.0+cpu e NumPy 2.3.5, CPU, em 9 de setembro de 2026. Não requer GPU, arquivos externos nem credenciais. Reinicie e execute todas as células em ordem. Os outputs ficam limpos no repositório; os números de referência estão na aula.

Dependências de validação do arquivo: nbformat >=5.10; não é necessário importá-lo para estudar. A fixture é algébrica, gerada com seed 20260906, sem estimação de desempenho ou escolha por conjunto de teste.

In [ ]:
import copy
import sys
import numpy as np
import torch
from torch import nn

SEED = 20260906
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
dtype = torch.float64
checks = []
def check(name, condition):
    assert bool(condition), name
    checks.append(name)

print('Python', sys.version.split()[0], '| torch', torch.__version__, '| NumPy', np.__version__)
check('CPU', torch.empty(0).device.type == 'cpu')

## 1. A mesma MLP, antes de introduzir a classe

Cinco exemplos, três entradas, quatro unidades ocultas e duas saídas. Os pesos têm orientação `(entrada, saída)`, como no cálculo manual. A loss é metade do erro quadrático médio sobre os dez resíduos. Todas as comparações usam cópias dos mesmos valores; seeds iguais entre bibliotecas não garantem amostras iguais.

In [ ]:
X = torch.tensor(rng.normal(size=(5, 3)), dtype=dtype)
T = torch.tensor(rng.normal(size=(5, 2)), dtype=dtype)
fixture = {
    'W1': torch.tensor(rng.normal(scale=.2, size=(3, 4)), dtype=dtype),
    'b1': torch.tensor(rng.normal(scale=.1, size=4), dtype=dtype),
    'W2': torch.tensor(rng.normal(scale=.2, size=(4, 2)), dtype=dtype),
    'b2': torch.tensor(rng.normal(scale=.1, size=2), dtype=dtype),
}
def explicit(x, p):
    return torch.tanh(x @ p['W1'] + p['b1']) @ p['W2'] + p['b2']
plain = {k: v.clone().requires_grad_() for k, v in fixture.items()}
out_plain = explicit(X, plain)
loss_plain = ((out_plain - T)**2).mean() / 2
grads_plain = torch.autograd.grad(loss_plain, tuple(plain.values()))
check('26 escalares', sum(p.numel() for p in plain.values()) == 26)
check('shape da saída', out_plain.shape == (5, 2))
print('Loss explícita:', f'{loss_plain.item():.12f}')

## 2. Encapsular estado preservando a função

`super().__init__()` prepara o registro. Atribuir `nn.Parameter` registra um peso; atribuir um módulo registra um filho. O backward continua sendo responsabilidade do autograd. As cópias evitam compartilhar acidentalmente a fixture.

In [ ]:
class Affine(nn.Module):
    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(weight.detach().clone())
        self.bias = nn.Parameter(bias.detach().clone())

    def forward(self, x):
        return x @ self.weight + self.bias

class MLP(nn.Module):
    def __init__(self, p):
        super().__init__()
        self.hidden = Affine(p['W1'], p['b1'])
        self.output = Affine(p['W2'], p['b2'])

    def forward(self, x):
        return self.output(torch.tanh(self.hidden(x)))

model = MLP(fixture)
out_module = model(X)
loss_module = ((out_module - T)**2).mean() / 2
grads_module = torch.autograd.grad(loss_module, tuple(model.parameters()))
forward_error = (out_module - out_plain).abs().max().item()
gradient_error = max((a-b).abs().max().item() for a,b in zip(grads_module, grads_plain))
check('paridade forward', forward_error < 1e-14)
check('paridade gradientes', gradient_error < 1e-14)
check('paridade loss', torch.allclose(loss_module, loss_plain, atol=1e-14, rtol=0))
check('parâmetros leaf', all(p.is_leaf for p in model.parameters()))
print('Erro forward:', forward_error, '| erro gradientes:', gradient_error)

## 3. Inventário, hierarquia e contagem

`parameters()` percorre recursivamente; `recurse=False` limita ao módulo atual. A raiz MLP possui filhos, mas nenhum parâmetro diretamente registrado nela. Não confunda quatro objetos Parameter com 26 números treináveis.

In [ ]:
names = list(dict(model.named_parameters()))
check('nomes esperados', names == ['hidden.weight', 'hidden.bias', 'output.weight', 'output.bias'])
check('raiz sem parâmetros diretos', len(list(model.parameters(recurse=False))) == 0)
check('dois filhos', list(dict(model.named_children())) == ['hidden', 'output'])
check('modules inclui raiz', list(dict(model.named_modules())) == ['', 'hidden', 'output'])
check('26 números registrados', sum(p.numel() for p in model.parameters()) == 26)
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel(), 'requires_grad=', p.requires_grad)
shared = nn.Module()
shared.a = nn.Parameter(torch.ones(3, dtype=dtype))
shared.b = shared.a
check('parâmetro compartilhado deduplicado', len(list(shared.parameters())) == 1)
check('dois nomes no estado compartilhado', set(shared.state_dict()) == {'a', 'b'})

## 4. Contraprova: gradiente existe, registro não

Um Tensor comum com `requires_grad=True` participa do autograd. Isso não o transforma em Parameter. Uma rotina que recebe apenas `model.parameters()` não o verá.

In [ ]:
class Unregistered(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = torch.tensor([2.0], dtype=dtype, requires_grad=True)
    def forward(self, x):
        return x * self.weight

bad = Unregistered()
bad(torch.tensor([3.0], dtype=dtype)).sum().backward()
check('gradiente no tensor comum', bad.weight.grad.item() == 3)
check('tensor comum invisível em parameters', len(list(bad.parameters())) == 0)
check('tensor comum fora do estado', len(bad.state_dict()) == 0)
good = nn.Module()
good.weight = nn.Parameter(bad.weight.detach().clone())
check('Parameter registrado', list(dict(good.named_parameters())) == ['weight'])
print('Tensor comum: gradiente =', bad.weight.grad.item(), '; parâmetros registrados =', len(list(bad.parameters())))

## 5. Buffers: estado que não é peso de otimização

O exemplo usa centro e escala **fixados pelo protocolo**, sem ajuste nos dados. Em um pipeline real, estatísticas de padronização seriam ajustadas somente no treino. O buffer não persistente é uma constante reconstruível; ele acompanha conversões, mas não entra no state_dict.

In [ ]:
class FixedScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('center', torch.tensor([1., 2., 3.], dtype=dtype))
        self.register_buffer('scale', torch.tensor([2., 4., 5.], dtype=dtype))
        self.register_buffer('offset', torch.zeros(3, dtype=dtype), persistent=False)
        self.note = torch.ones(3, dtype=dtype)  # atributo comum, não usado no forward
        self.description = 'escala fixa'
    def forward(self, x):
        return (x - self.center) / self.scale + self.offset

scale = FixedScale()
check('nenhum parâmetro no pré-processamento', len(list(scale.parameters())) == 0)
check('três buffers', set(dict(scale.named_buffers())) == {'center', 'scale', 'offset'})
check('apenas buffers persistentes no estado', set(scale.state_dict()) == {'center', 'scale'})
check('cálculo da escala', torch.equal(scale(torch.tensor([[3., 6., 8.]], dtype=dtype)), torch.ones(1, 3, dtype=dtype)))
converted = copy.deepcopy(scale).to(dtype=torch.float32)
check('buffers convertidos', all(b.dtype == torch.float32 for b in converted.buffers()))
check('atributo comum não convertido', converted.note.dtype == torch.float64)
model32 = copy.deepcopy(model).to(dtype=torch.float32)
check('parâmetros convertidos', all(p.dtype == torch.float32 for p in model32.parameters()))
check('modelo float32 executável', model32(X.float()).dtype == torch.float32)
print('Buffers:', list(dict(scale.named_buffers())), '| state_dict:', list(scale.state_dict()))

## 6. Listas Python versus contêineres registrados

Uma lista comum não registra recursivamente seus objetos. ModuleList registra módulos, mas não define como executá-los. ParameterList registra parâmetros. Sequential combina registro com encadeamento; implementamos a tanh sem antecipar a discussão das camadas prontas.

In [ ]:
class Stack(nn.Module):
    def __init__(self, registered):
        super().__init__()
        layers = [Affine(fixture['W1'], fixture['b1']), Affine(fixture['W2'], fixture['b2'])]
        self.layers = nn.ModuleList(layers) if registered else layers
    def forward(self, x):
        return self.layers[1](torch.tanh(self.layers[0](x)))

raw_stack, reg_stack = Stack(False), Stack(True)
check('lista comum executa a mesma função', torch.equal(raw_stack(X), reg_stack(X)))
check('lista comum não registra', len(list(raw_stack.parameters())) == 0)
check('ModuleList registra 26 escalares', sum(p.numel() for p in reg_stack.parameters()) == 26)
raw_stack.eval()
reg_stack.eval()
check('eval não alcança lista comum', raw_stack.layers[0].training)
check('eval alcança ModuleList', not reg_stack.layers[0].training)
holder = nn.Module()
holder.coefficients = nn.ParameterList([nn.Parameter(torch.ones(2, dtype=dtype))])
check('ParameterList registra', list(dict(holder.named_parameters())) == ['coefficients.0'])
class Tanh(nn.Module):
    def forward(self, x):
        return torch.tanh(x)
sequence = nn.Sequential(copy.deepcopy(model.hidden), Tanh(), copy.deepcopy(model.output))
check('Sequential mesma saída', torch.equal(sequence(X), model(X)))
print('Escalares na lista comum:', sum(p.numel() for p in raw_stack.parameters()), '| ModuleList:', sum(p.numel() for p in reg_stack.parameters()))

## 7. Congelamento, eval e no_grad respondem a perguntas diferentes

Congelar um parâmetro não o remove do módulo. Limpe gradientes anteriores antes de decidir quais pesos atualizar. Nesta MLP sem comportamento dependente do modo, eval não muda os números nem desativa o autograd.

In [ ]:
frozen = copy.deepcopy(model)
frozen.zero_grad(set_to_none=True)
frozen.hidden.requires_grad_(False)
check('congelados continuam registrados', sum(p.numel() for p in frozen.parameters()) == 26)
trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
check('dez escalares treináveis', trainable == 10)
frozen(X).sum().backward()
check('camada congelada sem gradiente novo', all(p.grad is None for p in frozen.hidden.parameters()))
check('saída recebe gradientes', all(p.grad is not None for p in frozen.output.parameters()))
frozen.eval()
check('eval recursivo', all(not m.training for m in frozen.modules()))
check('eval não desliga autograd', frozen(X).requires_grad)
with torch.no_grad():
    out_no_grad = frozen(X)
check('no_grad não registra operações', not out_no_grad.requires_grad)
check('no_grad não descongela parâmetros', not frozen.hidden.weight.requires_grad)
print('Registrados: 26; treináveis após congelar hidden:', trainable)

## 8. Chamar a instância preserva hooks

Um hook de observação registra somente o shape, sem guardar tensores com grafo. A chamada direta a forward ignora o hook instalado na raiz. Removemos o hook ao final para não alterar experimentos seguintes.

In [ ]:
events = []
def record_shape(module, args, output):
    events.append(tuple(output.shape))
handle = model.register_forward_hook(record_shape)
via_call = model(X)
via_forward = model.forward(X)
handle.remove()
check('saídas iguais nas duas chamadas', torch.equal(via_call, via_forward))
check('somente chamada à instância dispara hook da raiz', events == [(5, 2)])
model(X)
check('hook removido', len(events) == 1)
print('Eventos do hook:', events)

## 9. state_dict é um inventário com referências

Inspecionamos estado em memória; persistência em disco e checkpoints ficam nas Aulas 17–18. O dicionário padrão contém tensores destacados, mas compartilha armazenamento com o módulo. Uma cópia profunda congela a fotografia destes tensores. Não há snapshot automático de gradientes, flags, código ou configuração.

In [ ]:
probe = copy.deepcopy(model)
state = probe.state_dict()
snapshot = copy.deepcopy(state)
check('quatro chaves de estado', set(state) == set(names))
check('estado destacado por padrão', all(not t.requires_grad for t in state.values()))
check('estado compartilha storage', state['hidden.weight'].data_ptr() == probe.hidden.weight.data_ptr())
before = snapshot['hidden.weight'].clone()
with torch.no_grad():
    probe.hidden.weight.add_(1)
check('referência acompanha mutação', torch.equal(state['hidden.weight'], probe.hidden.weight))
check('cópia profunda preservada', torch.equal(snapshot['hidden.weight'], before))
state_delta = (state['hidden.weight'] - snapshot['hidden.weight']).abs().max().item()
check('mudança de uma unidade', abs(state_delta - 1) < 1e-14)
check('gradientes não são chaves', all(not k.endswith('.grad') for k in state))
print('Mudança observada no state_dict sem cópia:', state_delta)

## 10. Identidade dos parâmetros durante atualização

Uma lista capturada antes de substituir um Parameter continua apontando para o objeto antigo. Uma atualização numérica com copy_ dentro de no_grad mantém o objeto. O exemplo usa referências comuns para tornar o risco observável sem antecipar torch.optim.

In [ ]:
identity = copy.deepcopy(model)
captured = list(identity.parameters())
old_weight = identity.hidden.weight
identity.hidden.weight = nn.Parameter(old_weight.detach().clone())
check('substituição cria outra identidade', identity.hidden.weight is not old_weight)
check('lista anterior conserva objeto antigo', captured[0] is old_weight)
check('registro agora aponta ao novo objeto', next(identity.parameters()) is identity.hidden.weight)
current = identity.hidden.weight
with torch.no_grad():
    identity.hidden.weight.copy_(torch.zeros_like(current))
check('copy_ preserva identidade', identity.hidden.weight is current)
check('copy_ altera valores', torch.count_nonzero(current).item() == 0)
print('Referência anterior acompanha substituição?', captured[0] is identity.hidden.weight)

## 11. Auditoria final

Estas verificações cobrem contratos de registro, paridade matemática e contraprovas. Não medem acurácia, generalização, velocidade de GPU ou compatibilidade com qualquer checkpoint. A fixture não exige split porque nenhum ajuste ou seleção de modelo é realizado.

In [ ]:
check('nomes dos checks únicos', len(checks) == len(set(checks)))
check('saídas finitas', torch.isfinite(out_module).all())
check('gradientes finitos', all(torch.isfinite(g).all() for g in grads_module))
print(f'{len(checks)}/{len(checks)} verificações aprovadas')
print('Loss:', f'{loss_module.item():.12f}')
print('Erros máximos forward / gradientes:', forward_error, gradient_error)
print('Nenhum treino ou teste de generalização foi executado.')

## Exercícios e respostas

1. Por que bad.weight recebe gradiente sem aparecer em parameters? Autograd acompanha operações; o registro depende de Parameter ou registro explícito.
2. Quantos números restam treináveis ao congelar hidden? `4*2+2=10`; os 16 números ocultos continuam no estado.
3. Troque uma lista comum por ModuleList. O forward muda? Não neste exemplo; inventário, conversão e propagação de modos passam a alcançar os filhos.
4. O buffer offset pode ser recuperado de state_dict? Não, pois persistent=False; o construtor precisa reconstruí-lo coerentemente.
5. eval impede obter gradientes? Não; use no_grad para suprimir o registro das operações durante a execução.
6. Por que a cópia rasa não conserva o peso antigo? Os tensores do estado compartilham armazenamento; use cópia independente quando precisar de snapshot.

## Referências e continuação

Documentação oficial PyTorch 2.6, verificada em 9 de setembro de 2026:

- [Modules](https://docs.pytorch.org/docs/2.6/notes/modules.html)
- [nn.Module](https://docs.pytorch.org/docs/2.6/generated/torch.nn.Module.html)
- [Parameter](https://docs.pytorch.org/docs/2.6/generated/torch.nn.parameter.Parameter.html)
- [ModuleList](https://docs.pytorch.org/docs/2.6/generated/torch.nn.ModuleList.html)

Próxima aula: **07 — Camadas lineares, ativações e inicialização**, conforme o currículo. Substituiremos o bloco afim manual com atenção à orientação dos pesos.